# BoxCommentator: LoRA fine-tune of Qwen2.5-1.5B

Trains an adapter on `finetune/data/train.jsonl` (300 examples written by a larger teacher model, already
committed), then benchmarks base vs adapter in the same session.

Notebook settings: Accelerator = **GPU T4 x1**, Internet = **On**, Environment = **Always use latest**.

Order: run the training cells first (no vLLM yet). Then install vLLM, **Run > Restart session**, and run the
serving cells. Files in `/kaggle/working` survive the restart, so the adapter does not need uploading anywhere.

In [ ]:
!nvidia-smi

In [ ]:
REPO_URL = "https://github.com/sambar21/BoxCommentator.git"   # private repo: https://<token>@github.com/sambar21/BoxCommentator.git
BRANCH = "main"
!git clone --depth 1 -b {BRANCH} {REPO_URL} repo
%cd repo
!pip install -q -r finetune/requirements.txt pyyaml openai

In [ ]:
import json, pathlib
for name in ["train", "eval"]:
    p = pathlib.Path(f"finetune/data/{name}.jsonl")
    assert p.exists(), f"{p} missing: run finetune/gen_data.py and commit finetune/data/"
    print(name, sum(1 for _ in open(p)), "examples")
print(json.dumps(json.load(open("finetune/data/manifest.json"))["train"], indent=1)[:400])

## Train (a few minutes on a T4)

In [ ]:
!python -m finetune.train_lora --data finetune/data --out /kaggle/working/boxer_lora

In [ ]:
import json, shutil
log = json.load(open("/kaggle/working/boxer_lora/train_log.json"))
print("eval loss:", log["eval_loss"])
shutil.make_archive("/kaggle/working/boxer_lora", "zip", "/kaggle/working/boxer_lora")
print("download boxer_lora.zip from the Output tab")

## Serve and benchmark the adapter (same session, after training)
1. Run the next cell (installs vLLM and removes the mismatched torchaudio).
2. **Run > Restart session** (do not re-run the clone or install cells).
3. Run the last two cells. The `lora` plan runs the base 1.5B and the adapter on identical fights.

In [ ]:
!pip install -q vllm pyyaml openai matplotlib
!pip uninstall -y torchaudio

In [ ]:
# After Run > Restart session:
%cd /kaggle/working/repo
!pkill -f "vllm serve"
!ls /kaggle/working/boxer_lora
!python -m serving.run_matrix --plan lora --adapter /kaggle/working/boxer_lora --fights 10

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/lora_results", "zip", "bench/results")
shutil.copy("/kaggle/working/boxer_lora/train_log.json", "/kaggle/working/train_log.json")
print("download lora_results.zip and train_log.json from the Output tab")